# DeployAI Architecture Walkthrough

This notebook demonstrates the core concepts of Phase 1 (ReAct Agent & Database Connectivity) and Phase 2 (Human-in-the-Loop & System Resilience) that power our `deploy-agent` architecture.

## Phase 1: Tool Binding & Logic Encapsulation
We provide atomic capabilities to the LLM by defining well-documented Python functions with the `@tool` decorator. The LLM understands the docstrings and types to intelligently pass arguments.

In [ ]:
from langchain_core.tools import tool

@tool
def mock_get_server_info(server_name: str) -> str:
    """Simulates our SQLAlchemy database connection looking up IP addresses."""
    db = {
        "prod-server": "172.20.0.4",
        "test-server": "test-server"
    }
    return db.get(server_name, "Server not found in DB.")

@tool
def mock_ssh_execute(server_ip: str, command: str) -> str:
    """Mock Paramiko SSH execution tool."""
    return f"Ran '{command}' successfully on {server_ip} (Mocked output)"

# These tools will be strictly bound to our LLM:
my_tools = [mock_get_server_info, mock_ssh_execute]

## Phase 2: LangGraph Checkpoints & Human-in-the-Loop
We use `create_react_agent` linked to a `MemorySaver()` checkpoint tracker. By intentionally setting the flag `interrupt_before=["tools"]`, we force the Graph to safely freeze before destructive capabilities execute. 

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# Using LangChain's Fake LLM so you can see logic loops without setting real API keys
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage, ToolCall

# Simulating the LLM deciding it wants to run `df -h` on the `prod-server` mapped IP
fake_ai_msg_with_tool = AIMessage(
    content="Let me check that disk usage for you.", 
    tool_calls=[ToolCall(name="mock_ssh_execute", args={"server_ip":"172.20.0.4", "command":"df -h"}, id="call_1")]
)
fake_llm = FakeMessagesListChatModel(responses=[fake_ai_msg_with_tool])

# Build the Graph Configuration
memory_store = MemorySaver()
graph = create_react_agent(
    fake_llm,
    tools=my_tools, 
    checkpointer=memory_store,
    interrupt_before=["tools"] # <-- THE CRITICAL HITL FLAG FOR SAFETY
)

### Simulating the Execution Loop
We mimic the exact backend logic running our `app.py` CLI workflow.

In [ ]:
config = {"configurable": {"thread_id": "walkthrough_session"}}

# 1. Initial Invoke (The User's Request)
print("[ 🟢 ] Starting Graph Execution...")
result = graph.invoke({"messages": [("user", "Check disk usage on my prod server")]}, config=config)

# 2. Verify State Interrupts (Simulating CLI Prompt)
state = graph.get_state(config)

if state.next and state.next[0] == "tools":
    print("\n[ 🛑 ] ACTION REQUIRED: Graph execution is safely paused.")
    
    # Extract tool call data to show user
    last_ai_msg = [m for m in state.values.get("messages", []) if m.type == "ai"][-1]
    tool_request = last_ai_msg.tool_calls[0]
    print(f"      Target: {tool_request['name']}()")
    print(f"      Payload: {tool_request['args']}")
    
    print("\n[ 💻 ] Simulating user typing 'Y' in terminal to approve...")
    
    # 3. Resume the Workflow by passing `None`
    print("\n[ 🤖 ] Resuming workflow execution...")
    final_result = graph.invoke(None, config=config)
    
    print("\n--- ✅ PIPELINE FINISHED ---")